# 04 - 选址预测模型

XGBoost 二分类 + SHAP 可解释性分析

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report, roc_curve
from xgboost import XGBClassifier
import shap, joblib, warnings; warnings.filterwarnings('ignore')
import asyncpg, asyncio
plt.rcParams['font.sans-serif'] = ['SimHei']; plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 从数据库构建训练数据（复用 scripts/train_model.py 的逻辑）
# 此处为简化版，直接从 store_poi_stats pivot
import sys; sys.path.append('..')
from scripts.config import DB_CONFIG

async def load_data():
    conn = await asyncpg.connect(**DB_CONFIG)
    rows = await conn.fetch('''
        SELECT s.id, s.city, 1 AS label,
            COALESCE(MAX(CASE WHEN sps.poi_category='metro' AND sps.radius=500 THEN sps.poi_count END),0) AS metro_500,
            COALESCE(MAX(CASE WHEN sps.poi_category='office' AND sps.radius=500 THEN sps.poi_count END),0) AS office_500,
            COALESCE(MAX(CASE WHEN sps.poi_category='mall' AND sps.radius=500 THEN sps.poi_count END),0) AS mall_500,
            COALESCE(MAX(CASE WHEN sps.poi_category='restaurant' AND sps.radius=500 THEN sps.poi_count END),0) AS restaurant_500,
            COALESCE(MAX(CASE WHEN sps.poi_category='cafe' AND sps.radius=500 THEN sps.poi_count END),0) AS cafe_500,
            COALESCE(MAX(CASE WHEN sps.poi_category='residential' AND sps.radius=500 THEN sps.poi_count END),0) AS residential_500,
            COALESCE(MAX(CASE WHEN sps.poi_category='university' AND sps.radius=500 THEN sps.poi_count END),0) AS university_500
        FROM stores s
        LEFT JOIN store_poi_stats sps ON s.id=sps.store_id AND sps.radius=500
        WHERE s.brand='luckin'
        GROUP BY s.id, s.city
    ''')
    await conn.close()
    return pd.DataFrame([dict(r) for r in rows])

df = asyncio.run(load_data())
print(f'Loaded: {len(df)} positive samples')
df.head()

In [ ]:
# 训练简化模型
feature_cols = ['metro_500', 'office_500', 'mall_500', 'restaurant_500', 'cafe_500', 'residential_500', 'university_500']
X = pd.concat([df[feature_cols], df[feature_cols] * 0.1], ignore_index=True)  # 模拟负样本
y = np.concatenate([np.ones(len(df)), np.zeros(len(df))])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
print(f'AUC: {roc_auc_score(y_test, y_proba):.4f}')
print(classification_report(y_test, y_pred, target_names=['不适合', '适合']))

In [ ]:
# 特征重要性
importance = pd.DataFrame({'feature': feature_cols, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)
plt.figure(figsize=(8, 5))
plt.barh(importance['feature'], importance['importance'], color='#1677ff')
plt.xlabel('Importance'); plt.title('Feature Importance - Site Selection Model')
plt.tight_layout(); plt.show()

In [ ]:
# SHAP 分析
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test.sample(min(200, len(X_test))))

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test.sample(min(200, len(X_test))), feature_names=feature_cols, show=True)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, 'b-', label=f'AUC={roc_auc_score(y_test, y_proba):.3f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve'); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# 保存模型
model_path = '../data/models/xgboost_location_model.pkl'
import os; os.makedirs(os.path.dirname(model_path), exist_ok=True)
joblib.dump(model, model_path)
print(f'Model saved: {model_path}')